# 06 · 夜间联合增强-去雾

注意力引导 gamma 提亮 + 去雾主干 + 夜间专用损失；随机张量验证前向与损失。

In [ ]:
import torch
import torch.nn as nn


class GammaEnhance(nn.Module):
    def __init__(self, dim, gamma=0.3):
        super().__init__()
        self.gamma = gamma
        self.attn = nn.Sequential(
            nn.Conv2d(dim, dim // 8, 1), nn.ReLU(inplace=True),
            nn.Conv2d(dim // 8, 3, 1), nn.Sigmoid())

    def forward(self, feat, dehazed):
        A = self.attn(feat)
        enhanced = dehazed.clamp(1e-6).pow(self.gamma)
        return (1 - A) * dehazed + A * enhanced


class NightDehazeNet(nn.Module):
    def __init__(self, dim=32, n_blocks=6, gamma=0.3):
        super().__init__()
        self.stem = nn.Conv2d(3, dim, 3, padding=1)
        self.body = nn.Sequential(*[
            nn.Sequential(nn.Conv2d(dim, dim, 3, padding=1, padding_mode="reflect"),
                          nn.ReLU(inplace=True)) for _ in range(n_blocks)])
        self.dehaze_head = nn.Conv2d(dim, 3, 3, padding=1)
        self.enhance = GammaEnhance(dim, gamma)

    def forward(self, x):
        feat = self.body(self.stem(x))
        dehazed = (x + self.dehaze_head(feat)).clamp(0, 1)
        return self.enhance(feat, dehazed).clamp(0, 1)


class NightDehazeLoss(nn.Module):
    def __init__(self, l1_w=1.0, color_w=0.3, dark_w=0.2):
        super().__init__()
        self.l1_w, self.color_w, self.dark_w = l1_w, color_w, dark_w
        self.l1 = nn.L1Loss()

    def forward(self, pred, gt):
        loss = self.l1_w * self.l1(pred, gt)
        color_loss = torch.abs(pred.mean(dim=(2, 3)) - gt.mean(dim=(2, 3))).mean()
        dark_mask = (1 - gt.mean(dim=1, keepdim=True))
        dark_loss = (torch.abs(pred - gt) * dark_mask).mean()
        return loss + self.color_w * color_loss + self.dark_w * dark_loss


In [ ]:
model = NightDehazeNet(dim=32, n_blocks=4)
crit = NightDehazeLoss()
x = torch.randn(1, 3, 256, 256)
gt = torch.randn(1, 3, 256, 256)
y = model(x)
print("输入:", x.shape, "输出:", y.shape, "损失:", crit(y, gt).item())
print("参数量: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
